# PyTorch Lightning: A First Look

In the previous notebook (`02_MLP_fashionMNIST.ipynb`) we built a training loop entirely from scratch. That loop had a lot of **boilerplate**: zeroing gradients, calling `.backward()`, calling `.optimizer.step()`, moving data to the device, switching between `model.train()` and `model.eval()`, tracking losses and accuracies... All of that code is the same in almost every project.

**PyTorch Lightning** is a lightweight framework built directly on top of PyTorch that separates this boilerplate from your actual research code. The idea is simple:
* You put your **model and learning logic** in a `LightningModule`.
* You put your **data loading** in a `LightningDataModule`.
* A `Trainer` object handles everything else (the training loop, device management, logging, callbacks, etc.).

The result: less code and much easier experimentation. 

## Setup

### Install and import

Lightning is not part of PyTorch itself; it is a separate package. Install it with `pip install lightning`. In this notebook we use the modern `lightning` package (version ≥ 2.0); older code may use `pytorch_lightning` instead — they are the same thing, just renamed.

In [ ]:
# !pip install lightning torchmetrics   # uncomment if not installed

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets
from torchvision.transforms import ToTensor

import lightning as pl               ## the main Lightning module
import torchmetrics                   ## convenient accuracy / metric computation

print(f"PyTorch version:   {torch.__version__}")
print(f"Lightning version: {pl.__version__}")

### A note on device management

In the raw training loop (MLP_demo) we had to explicitly create a `device` variable and call `.to(device)` on the model and every batch. **Lightning handles this automatically**: the `Trainer` detects the best available hardware and moves your model and data there without you needing to do anything. You just write `pl.Trainer(accelerator='auto')` and Lightning figures out whether to use CUDA, MPS, or CPU.

## The LightningDataModule

A `LightningDataModule` encapsulates all data-related code: downloading, transforming, splitting into train/validation/test sets, and creating `DataLoader` objects. This keeps data logic completely separate from model logic.

The methods you need to implement:
* `setup(stage)`: called by the Trainer before training starts. This is where you load/split the dataset. The `stage` argument is `'fit'` (for training), `'validate'`, `'test'`, or `'predict'` — but you can often ignore it.
* `train_dataloader()`: returns the `DataLoader` for training.
* `val_dataloader()`: returns the `DataLoader` for validation.

We use FashionMNIST, the same dataset as in MLP_demo. For simplicity, we use the test set as the validation set (in a real project you would hold out a validation set from the training data).

In [ ]:
class FashionMNISTDataModule(pl.LightningDataModule):

    def __init__(self, batch_size=128):
        super().__init__()
        self.batch_size = batch_size       ## store batch size for use in dataloaders

    def setup(self, stage=None):
        ## Load the train and test splits; the transform converts PIL images to tensors
        self.train_data = datasets.FashionMNIST(
            root="data", train=True,  download=True, transform=ToTensor()
        )
        self.val_data = datasets.FashionMNIST(
            root="data", train=False, download=True, transform=ToTensor()
        )
        ## Note: we use the official test split as our validation set here.
        ## In practice you'd do: self.train_data, self.val_data = random_split(full_train, [55000, 5000])

    def train_dataloader(self):
        ## shuffle=True: crucial for training (prevents the model from memorising the order)
        return DataLoader(self.train_data, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        ## shuffle=False for validation: order doesn't matter and reproducibility is nice
        return DataLoader(self.val_data, batch_size=self.batch_size)

## The LightningModule

A `LightningModule` is the heart of a Lightning project. It inherits from both `nn.Module` and the Lightning base class. Think of it as the raw PyTorch model PLUS all the training/validation logic that in MLP_demo lived in the `train()` and `test()` functions.

Here is the mapping from MLP_demo to Lightning:

| MLP_demo | LightningModule |
|---|---|
| `__init__` of `NeuralNetwork` | `__init__` of `LightningModule` |
| `forward` | `forward` |
| One iteration of the `train()` function | `training_step()` |
| One iteration of the `test()` function | `validation_step()` |
| `optimizer = torch.optim.SGD(...)` | `configure_optimizers()` |
| `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` | **handled automatically by the Trainer** |
| `X, y = X.to(device), y.to(device)` | **handled automatically by the Trainer** |
| `model.train()` / `model.eval()` | **handled automatically by the Trainer** |

Notice what disappears: no more zero_grad, backward, step, or device management in your code.

In [ ]:
class FashionMNISTClassifier(pl.LightningModule):

    def __init__(self, lr=1e-2):
        super().__init__()
        self.lr = lr                ## store learning rate for use in configure_optimizers

        ## Network architecture: same as MLP_demo
        self.net = nn.Sequential(
            nn.Flatten(),              ## flatten 4D batch (B, 1, 28, 28) → 2D (B, 784)
            nn.Linear(28*28, 512),     ## first hidden layer: 784 inputs, 512 outputs
            nn.ReLU(),
            nn.Linear(512, 512),       ## second hidden layer: keeps the size
            nn.ReLU(),
            nn.Linear(512, 10),        ## output layer: 10 classes
        )

        self.loss_fn = nn.CrossEntropyLoss()   ## includes softmax; outputs logits are expected

        ## torchmetrics.Accuracy handles the argmax and averaging for us
        ## 'multiclass' means more-than-2 classes, num_classes=10 specifies the number
        self.train_acc = torchmetrics.Accuracy(task='multiclass', num_classes=10)
        self.val_acc   = torchmetrics.Accuracy(task='multiclass', num_classes=10)

    def forward(self, x):
        ## Forward pass: same as in MLP_demo
        return self.net(x)

    def training_step(self, batch, batch_idx):
        ## batch is a (X, y) tuple; batch_idx is the index of this batch (sometimes useful)
        ## Lightning automatically puts X and y on the right device — no .to(device) needed
        X, y = batch
        logits = self(X)                      ## self(X) calls forward(X)
        loss   = self.loss_fn(logits, y)      ## compute cross-entropy loss

        ## self.log() sends the value to all configured loggers (TensorBoard, CSV, etc.)
        ## on_epoch=True: average the value over the epoch and show it at the end
        ## prog_bar=True: show it in the tqdm progress bar during training
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)

        ## Update the training accuracy metric
        self.train_acc(logits, y)
        self.log('train_acc', self.train_acc, on_step=False, on_epoch=True, prog_bar=True)

        ## IMPORTANT: return the loss so Lightning can call .backward() and .step() on it
        return loss

    def validation_step(self, batch, batch_idx):
        ## Identical structure to training_step, but:
        ##   - no need to return the loss
        ##   - Lightning has already called model.eval() and torch.no_grad() for us
        X, y = batch
        logits = self(X)
        loss   = self.loss_fn(logits, y)

        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.val_acc(logits, y)
        self.log('val_acc', self.val_acc, on_step=False, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        ## Return the optimizer. This is called once by the Trainer before training starts.
        ## Lightning will call optimizer.zero_grad(), optimizer.step() automatically.
        return torch.optim.SGD(self.parameters(), lr=self.lr)

## The Trainer

The `Trainer` is the object that orchestrates everything. You tell it *how* you want to train (how many epochs, which accelerator, which callbacks, which logger, etc.) and it takes care of the rest.

Common `Trainer` arguments:
* `max_epochs`: total number of training epochs.
* `accelerator`: `'auto'` (default, picks the best available), `'gpu'`, `'mps'`, `'cpu'`.
* `logger`: which logging backend to use (default: TensorBoard if installed, or CSV).
* `callbacks`: a list of `Callback` objects (see below).
* `log_every_n_steps`: how often to log metrics to the logger (default: 50 steps).

Then you call `trainer.fit(model, datamodule)` to start training.

In [ ]:
## Instantiate the data module and the model
datamodule = FashionMNISTDataModule(batch_size=128)
model      = FashionMNISTClassifier(lr=1e-2)

## Create the Trainer
trainer = pl.Trainer(
    max_epochs=5,            ## train for 5 epochs
    accelerator='auto',      ## pick GPU/MPS/CPU automatically
    log_every_n_steps=10,    ## log metrics every 10 training steps
)

print(model)    ## Lightning prints a nice model summary automatically during fit,
                ## but you can also preview the model structure here

In [ ]:
## Start training! Compare this one line to the entire loop in MLP_demo.
trainer.fit(model, datamodule=datamodule)

After running the cell above you should see:
* A model summary table (layer names, output shapes, number of parameters).
* A progress bar per epoch showing `train_loss`, `train_acc`, `val_loss`, `val_acc`.
* A final "Epoch X finished" message.

Notice that Lightning ran the validation loop automatically after each training epoch. You never had to write the code to switch the model to eval mode, disable gradients, or compute the test accuracy — the Trainer did all of that.

## Running inference after training

After training, using the model for inference is the same as raw PyTorch: get a sample, call `model.eval()`, wrap in `torch.no_grad()`, and call `model(x)`.

In [ ]:
classes = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal",      "Shirt",   "Sneaker",  "Bag",   "Ankle boot"
]

## Get a sample from the validation set; note the DataModule must have been set up first
datamodule.setup()     ## trigger data loading if not already done
val_dataset = datamodule.val_data

x, y_true = val_dataset[0]           ## x is a (1, 28, 28) tensor, y_true is an integer
x = x.unsqueeze(0)                   ## add batch dimension: (1, 28, 28) → (1, 1, 28, 28)

model.eval()
with torch.no_grad():
    logits = model(x)                ## forward pass; output shape: (1, 10)
    pred   = logits.argmax(dim=1)    ## class with highest score

print(f"True class:      {classes[y_true]}")
print(f"Predicted class: {classes[pred.item()]}")

## Callbacks: model checkpointing and early stopping

Callbacks are objects that hook into the training loop at predefined points (end of epoch, end of validation, etc.) and perform side effects like saving the model, stopping training early, or adjusting the learning rate. Lightning comes with many built-in callbacks.

### ModelCheckpoint

Automatically saves the model weights to disk whenever a monitored metric improves. This is important for long training runs: if training crashes or you want to use the best model (not the last one), you can load the saved checkpoint.

### EarlyStopping

Stops training automatically when a monitored metric stops improving for `patience` consecutive epochs. This prevents overfitting and saves time.

In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

## Save the best model (by validation accuracy)
checkpoint_callback = ModelCheckpoint(
    monitor='val_acc',       ## which metric to watch
    mode='max',              ## 'max' because higher accuracy is better
    save_top_k=1,            ## save only the single best checkpoint
    verbose=True,            ## print a message when saving
    filename='best-{epoch:02d}-{val_acc:.3f}',  ## checkpoint filename template
)

## Stop if val_acc hasn't improved for 3 consecutive epochs
early_stopping = EarlyStopping(
    monitor='val_acc',   ## which metric to watch
    patience=3,          ## number of epochs without improvement before stopping
    mode='max',          ## 'max' because higher accuracy is better
    verbose=True,
)

## Rebuild model and trainer with callbacks
model2   = FashionMNISTClassifier(lr=1e-2)
trainer2 = pl.Trainer(
    max_epochs=50,
    accelerator='auto',
    log_every_n_steps=10,
    callbacks=[checkpoint_callback, early_stopping],  ## pass callbacks here
)

trainer2.fit(model2, datamodule=datamodule)

In [ ]:
## The path to the best saved checkpoint
print(f"Best model saved at: {checkpoint_callback.best_model_path}")
print(f"Best val_acc:        {checkpoint_callback.best_model_score:.4f}")

### Loading a saved checkpoint

There are two common reasons to load a checkpoint:

* **Resume training** from where you left off (e.g. after a crash or to continue for more epochs).
* **Load the best model for inference**, rather than using the last model state which may be slightly worse.

Lightning provides `load_from_checkpoint`, a class method that re-creates the model with the saved weights. The key requirement is that the `__init__` signature of your `LightningModule` must match what was used when the checkpoint was saved (Lightning stores the hyperparameters automatically if you call `self.save_hyperparameters()` in `__init__`, but that is optional).

In [ ]:
## --- Option 1: load the best checkpoint for inference ---
best_ckpt_path = checkpoint_callback.best_model_path   ## path saved by ModelCheckpoint

## load_from_checkpoint re-creates the model and restores the saved weights
## The model is returned in eval mode, ready for predictions
loaded_model = FashionMNISTClassifier.load_from_checkpoint(best_ckpt_path)
print(f"Loaded model from: {best_ckpt_path}")
print(f"val_acc of this checkpoint: {checkpoint_callback.best_model_score:.4f}")
print()

## Use it for inference exactly as you would any other model
datamodule.setup()
x, y_true = datamodule.val_data[0]
x = x.unsqueeze(0)          ## add batch dimension
loaded_model.eval()
with torch.no_grad():
    pred = loaded_model(x).argmax(dim=1).item()

classes = ["T-shirt/top","Trouser","Pullover","Dress","Coat",
           "Sandal","Shirt","Sneaker","Bag","Ankle boot"]
print(f"True: {classes[y_true]}  |  Predicted: {classes[pred]}")
print()

## --- Option 2: resume training from a checkpoint ---
## Pass ckpt_path to trainer.fit() and Lightning restores weights, optimizer
## state, epoch counter, and logged metrics — everything picks up from where it left off.
##
##   trainer_new = pl.Trainer(max_epochs=100, ...)   # train for more epochs total
##   trainer_new.fit(model, datamodule, ckpt_path=best_ckpt_path)
##
## Note: ckpt_path='last' is a shorthand to resume from the most recent checkpoint
## (ModelCheckpoint saves a 'last.ckpt' automatically unless you disable it).
print("To resume training, pass ckpt_path to trainer.fit():")
print("  trainer.fit(model, datamodule, ckpt_path='path/to/checkpoint.ckpt')")

## Distributed training strategies

One of the most powerful features of the Lightning `Trainer` is that it can distribute training across multiple GPUs or machines with virtually no changes to your code. You just pass a `strategy` argument. You will almost certainly not use these during this course (you have one GPU at most), but it is worth knowing they exist, because at some point you will need them.

### Data parallelism — `strategy='ddp'`

The most common approach. The model is **replicated** on each GPU. Each GPU receives a different slice of the mini-batch, computes its forward and backward passes independently, then the gradients are averaged across all GPUs before the parameter update. Effectively this multiplies your batch size by the number of GPUs while keeping each GPU busy.

```
GPU 0: model copy  ← batch slice 0
GPU 1: model copy  ← batch slice 1   → sync gradients → update all copies identically
GPU 2: model copy  ← batch slice 2
```

`'ddp'` stands for **Distributed Data Parallel**, PyTorch's recommended multi-GPU backend. There is also the older `'dp'` (Data Parallel) which works within a single machine but is slower and has subtle issues with `self.log`; prefer `'ddp'`.

### Model parallelism

Used when the model itself is **too large to fit on a single GPU**. Different layers (or tensor shards) are placed on different GPUs. This is the regime of very large language models. Lightning supports this via strategies like `'fsdp'` (Fully Sharded Data Parallel) and integrations with DeepSpeed.

### In practice

```python
# Single GPU (default, what you will use)
trainer = pl.Trainer(accelerator='auto', devices=1)

# All available GPUs on one machine, DDP
trainer = pl.Trainer(accelerator='gpu', devices=-1, strategy='ddp')

# 4 GPUs explicitly
trainer = pl.Trainer(accelerator='gpu', devices=4, strategy='ddp')

# Multiple machines (nodes): 3 nodes × 8 GPUs each = 24 GPUs total
trainer = pl.Trainer(accelerator='gpu', devices=8, num_nodes=3, strategy='ddp')
```

Your `LightningModule` code — `training_step`, `forward`, `configure_optimizers` — does not change at all. Lightning handles the synchronisation, gradient averaging, and process launching behind the scenes.